# Anomaly Vectorization Comparison

This notebook compares two anomaly feature spaces:

- Sparse char-level TF-IDF
- Dense TF-IDF + LSA

The goal is to check whether LSA improves anomaly separation quality.

In [11]:
from pathlib import Path

import pandas as pd

from data_mining_assignment.core.data_io import ArticleDataset
from data_mining_assignment.tasks.anomaly_detection import TextAnomalyDetector
from data_mining_assignment.tasks.exploration import build_anomaly_candidate_table
from data_mining_assignment.tasks.preprocessing import TextNormalizer, TextPreprocessor

## Load and normalize data

In [12]:
project_root_path = Path.cwd().parent
articles_csv_path = project_root_path / "data" / "raw" / "articles.csv"
articles_data_frame = ArticleDataset(input_csv_path=articles_csv_path).load_articles()

document_ids = articles_data_frame["doc_id"].tolist()
raw_texts = articles_data_frame["text"].tolist()
normalized_bundle = TextNormalizer().normalize_for_both_tasks(raw_texts)

## Model A: Sparse TF-IDF

In [13]:
sparse_preprocessor = TextPreprocessor(
    vectorization_model_name="tfidf",
    max_features=30000,
    min_document_frequency=1,
    max_document_frequency=1.0,
    ngram_range=(3, 5),
    analyzer_mode="char_wb",
)
sparse_features = sparse_preprocessor.fit_transform(normalized_bundle.anomaly_texts)

sparse_detector = TextAnomalyDetector(contamination_ratio=0.02, random_seed=42)
sparse_mask, sparse_scores = sparse_detector.run_detection(sparse_features)
sparse_table = build_anomaly_candidate_table(document_ids, sparse_scores.tolist(), sparse_mask.tolist())

## Model B: Dense TF-IDF + LSA

In [14]:
lsa_preprocessor = TextPreprocessor(
    vectorization_model_name="tfidf_lsa_dense",
    max_features=30000,
    min_document_frequency=1,
    max_document_frequency=1.0,
    ngram_range=(3, 5),
    analyzer_mode="char_wb",
    dense_embedding_dimension=256,
    random_seed=42,
)
lsa_features = lsa_preprocessor.fit_transform(normalized_bundle.anomaly_texts)

lsa_detector = TextAnomalyDetector(contamination_ratio=0.02, random_seed=42)
lsa_mask, lsa_scores = lsa_detector.run_detection(lsa_features)
lsa_table = build_anomaly_candidate_table(document_ids, lsa_scores.tolist(), lsa_mask.tolist())

## Compare top anomalies

In [15]:
comparison_table = pd.DataFrame(
    {
        "rank": range(1, 21),
        "sparse_doc_id": sparse_table.head(20)["doc_id"].tolist(),
        "lsa_doc_id": lsa_table.head(20)["doc_id"].tolist(),
        "sparse_score": sparse_table.head(20)["score"].tolist(),
        "lsa_score": lsa_table.head(20)["score"].tolist(),
    }
)
comparison_table

,rank,sparse_doc_id,lsa_doc_id,sparse_score,lsa_score
0,1,DOC_00746,DOC_01750,-0.392687,-0.504923
1,2,DOC_01390,DOC_00909,-0.392687,-0.501745
2,3,DOC_01877,DOC_00821,-0.391215,-0.482865
3,4,DOC_00847,DOC_01025,-0.376699,-0.480890
4,5,DOC_01607,DOC_00443,-0.374241,-0.479748
5,6,DOC_00228,DOC_01211,-0.373499,-0.479338
6,7,DOC_01038,DOC_00223,-0.373197,-0.478795
7,8,DOC_01363,DOC_01833,-0.370283,-0.477093
8,9,DOC_00873,DOC_02135,-0.366945,-0.476993
9,10,DOC_01044,DOC_01114,-0.366686,-0.476229


## Overlap diagnostics

In [16]:
def compute_top_k_overlap(
    sparse_ranked_table: pd.DataFrame,
    lsa_ranked_table: pd.DataFrame,
    top_k: int,
) -> int:
    """Computes overlap between two ranked top-k anomaly sets."""
    sparse_top_ids = set(sparse_ranked_table.head(top_k)["doc_id"].tolist())
    lsa_top_ids = set(lsa_ranked_table.head(top_k)["doc_id"].tolist())
    return len(sparse_top_ids.intersection(lsa_top_ids))


overlap_diagnostics = pd.DataFrame(
    {
        "top_k": [10, 25, 50, 75, 100],
        "overlap_count": [
            compute_top_k_overlap(sparse_table, lsa_table, top_k_value) for top_k_value in [10, 25, 50, 75, 100]
        ],
    }
)
overlap_diagnostics

,top_k,overlap_count
0,10,0
1,25,0
2,50,0
3,75,0
4,100,0


## Why overlap can be low

In [17]:
from scipy.stats import spearmanr

score_rank_correlation = spearmanr(sparse_scores, lsa_scores).correlation

summary_diagnostics = {
    "sparse_predicted_anomaly_count": int(sparse_mask.sum()),
    "lsa_predicted_anomaly_count": int(lsa_mask.sum()),
    "spearman_score_correlation": float(score_rank_correlation),
}
summary_diagnostics

{'sparse_predicted_anomaly_count': 44,
 'lsa_predicted_anomaly_count': 44,
 'spearman_score_correlation': -0.2416820870035977}

## Overlap score in top-50

In [18]:
sparse_top_50 = set(sparse_table.head(50)["doc_id"].tolist())
lsa_top_50 = set(lsa_table.head(50)["doc_id"].tolist())
overlap_count = len(sparse_top_50.intersection(lsa_top_50))
overlap_count

0